<a href="https://colab.research.google.com/github/suleman12344/spam-emails/blob/main/spam_email_detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import kagglehub
path = kagglehub.dataset_download("ashfakyeafi/spam-email-classification")

100%|██████████| 207k/207k [00:00<00:00, 56.9MB/s]

Extracting files...


In [95]:
import pandas as pd
df = pd.read_csv(path+"/email.csv")

In [96]:
print(df.head(5))
print(df.shape)
df["Category"].value_counts()

  Category                                            Message
0      ham  Go until jurong point, crazy.. Available only ...
1      ham                      Ok lar... Joking wif u oni...
2     spam  Free entry in 2 a wkly comp to win FA Cup fina...
3      ham  U dun say so early hor... U c already then say...
4      ham  Nah I don't think he goes to usf, he lives aro...
(5573, 2)


,count
Category,
ham,4825
spam,747
"{""mode"":""full""",1


In [97]:
df = pd.get_dummies(df,columns=['Category'], drop_first=True ,dtype=int)
df = df.drop(df['Category_{"mode":"full"'])
print(df.head(5))
print(df.shape)

                                             Message  Category_spam  \
2  Free entry in 2 a wkly comp to win FA Cup fina...              1   
3  U dun say so early hor... U c already then say...              0   
4  Nah I don't think he goes to usf, he lives aro...              0   
5  FreeMsg Hey there darling it's been 3 week's n...              1   
6  Even my brother is not like to speak with me. ...              0   

   Category_{"mode":"full"  
2                        0  
3                        0  
4                        0  
5                        0  
6                        0  
(5571, 3)


In [98]:
df = df.drop(columns=['Category_{"mode":"full"'])
print(df.head(5))

                                             Message  Category_spam
2  Free entry in 2 a wkly comp to win FA Cup fina...              1
3  U dun say so early hor... U c already then say...              0
4  Nah I don't think he goes to usf, he lives aro...              0
5  FreeMsg Hey there darling it's been 3 week's n...              1
6  Even my brother is not like to speak with me. ...              0


In [99]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB


vectorizer = CountVectorizer()
x = vectorizer.fit_transform(df['Message'])

In [100]:
x_train,x_test,y_train,y_test = train_test_split(x,df['Category_spam'],test_size=0.2)

In [104]:
model = MultinomialNB()
model.fit(x_train,y_train)
pred_y = model.predict(x_test)
model_score = model.score(x_test,pred_y)
print(model_score)

1.0


In [105]:
new_emails = [
    "Congratulations! You've won a free iPhone. Click here to claim your prize.",
    "Hey, just checking in to see if you received my last email about the meeting tomorrow."
]

X_new = vectorizer.transform(new_emails)

predictions = model.predict(X_new.toarray())
print(predictions)

[1 0]


In [106]:
from sklearn.pipeline import Pipeline

pipe = Pipeline([
    ('vectorizer', CountVectorizer()),
    ('model', MultinomialNB())
])

In [107]:
pipe.fit(df['Message'], df['Category_spam'])

Pipeline(steps=[('vectorizer', CountVectorizer()), ('model', MultinomialNB())])

In [108]:
new_emails_for_prediction = [
    "You have won a lottery of $1,000,000. Click here to claim.",
    "Hi John, just a reminder about our meeting tomorrow at 10 AM.",
    "Your account has been compromised. Verify your details now!"
]

pipeline_predictions = pipe.predict(new_emails_for_prediction)

for email, prediction in zip(new_emails_for_prediction, pipeline_predictions):
    status = 'spam' if prediction == 1 else 'ham'
    print(f"Email: '{email}'\nPrediction: {status}\n")

Email: 'You have won a lottery of $1,000,000. Click here to claim.'
Prediction: spam

Email: 'Hi John, just a reminder about our meeting tomorrow at 10 AM.'
Prediction: ham

Email: 'Your account has been compromised. Verify your details now!'
Prediction: spam

